In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt 
import pickle as pkl
from sklearn.model_selection  import train_test_split


In [2]:
book = pd.read_csv("Books.csv")
ratin = pd.read_csv("Ratings.csv")
user= pd.read_csv("Users.csv")

C:\Users\vivek\AppData\Local\Temp\ipykernel_15372\3371696119.py:1: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  book = pd.read_csv("Books.csv")


In [3]:
book.sample()

,ISBN,Book-Title,Book-Author,Year-Of-Publication,Publisher,Image-URL-S,Image-URL-M,Image-URL-L
214294,0140130187,Souls &amp; Bodies (King Penguin),David Lodge,1989,Penguin Books,http://images.amazon.com/images/P/0140130187.0...,http://images.amazon.com/images/P/0140130187.0...,http://images.amazon.com/images/P/0140130187.0...


In [4]:
books = book.drop(["Image-URL-S", "Image-URL-M","Image-URL-L"],axis=1)

In [5]:
books.sample(5)

,ISBN,Book-Title,Book-Author,Year-Of-Publication,Publisher
207164,0670868035,With Custer on the Little Bighorn: A Newly Dis...,William O. Taylor,1996,Viking Books
223286,3150181305,Troia: Mythos und Wirklichkeit (Universal-Bibl...,Michael Siebler,2001,Reclam
177771,0006176747,Medusa,Hammond Innes,0,Fontana Paperbacks
96515,0060266686,In the Night Kitchen (Caldecott Collection),Maurice Sendak,1996,HarperCollins
195312,0553226754,Chill,Ross MacDonald,1983,Bantam Books


In [6]:
books = books.rename(columns={'Book-Title':'name','Book-Author':'author','Year-Of-Publication':'year','publisher':'Publisher'})

In [7]:
books

,ISBN,name,author,year,Publisher
0,0195153448,Classical Mythology,Mark P. O. Morford,2002,Oxford University Press
1,0002005018,Clara Callan,Richard Bruce Wright,2001,HarperFlamingo Canada
2,0060973129,Decision in Normandy,Carlo D'Este,1991,HarperPerennial
3,0374157065,Flu: The Story of the Great Influenza Pandemic...,Gina Bari Kolata,1999,Farrar Straus Giroux
4,0393045218,The Mummies of Urumchi,E. J. W. Barber,1999,W. W. Norton &amp; Company
...,...,...,...,...,...
271355,0440400988,There's a Bat in Bunk Five,Paula Danziger,1988,Random House Childrens Pub (Mm)
271356,0525447644,From One to One Hundred,Teri Sloat,1991,Dutton Books
271357,006008667X,Lily Dale : The True Story of the Town that Ta...,Christine Wicker,2004,HarperSanFrancisco
271358,0192126040,Republic (World's Classics),Plato,1996,Oxford University Press


In [8]:
rating=ratin.rename(columns={'User-ID':'id','Book-Rating':'rating'})

In [9]:
rating.groupby('ISBN').count()

,id,rating
ISBN,,
0330299891,2,2
0375404120,2,2
0586045007,1,1
9022906116,2,2
9032803328,1,1
...,...,...
cn113107,1,1
ooo7156103,1,1
§423350229,1,1


In [10]:
user.sample()

,User-ID,Location,Age
9857,9858,"pittsburgh, pennsylvania, usa",NaN


In [12]:
user=user.rename(columns={'User-ID':'id','Location':'location','Age':'age'})

In [13]:
x =rating['id'].value_counts() > 200

In [14]:
y=x[x].index

In [15]:
rating=rating[rating['id'].isin(y)]

In [16]:
rating.shape

(526356, 3)

In [17]:
rating_with_books=rating.merge(books,on='ISBN')

In [18]:
rating_with_books.shape

(487671, 7)

In [19]:
rating_with_books[rating_with_books['rating'].isin(y)]

,id,ISBN,rating,name,author,year,Publisher


In [126]:
number_rating=rating_with_books.groupby('name')['rating'].count().reset_index()

In [128]:
number_rating=number_rating.rename(columns={'rating':'num_rating'})

In [129]:
final_rating=rating_with_books.merge(number_rating,on='name')

In [130]:
final_rating.shape

(487671, 8)

In [131]:
final_rating=final_rating[final_rating['num_rating'] >= 50 ]

In [132]:
final_rating

,id,ISBN,rating,name,author,year,Publisher,num_rating
0,277427,002542730X,10,Politically Correct Bedtime Stories: Modern Ta...,James Finn Garner,1994,John Wiley &amp; Sons Inc,82
13,277427,0060930535,0,The Poisonwood Bible: A Novel,Barbara Kingsolver,1999,Perennial,133
15,277427,0060934417,0,Bel Canto: A Novel,Ann Patchett,2002,Perennial,108
18,277427,0061009059,9,One for the Money (Stephanie Plum Novels (Pape...,Janet Evanovich,1995,HarperTorch,108
24,277427,006440188X,0,The Secret Garden,Frances Hodgson Burnett,1998,HarperTrophy,79
...,...,...,...,...,...,...,...,...
487505,275970,1400031354,0,Tears of the Giraffe (No.1 Ladies Detective Ag...,Alexander McCall Smith,2002,Anchor,84
487506,275970,1400031362,0,Morality for Beautiful Girls (No.1 Ladies Dete...,Alexander McCall Smith,2002,Anchor,60
487579,275970,1573229725,0,Fingersmith,Sarah Waters,2002,Riverhead Books,59
487618,275970,1586210661,9,Me Talk Pretty One Day,David Sedaris,2001,Time Warner Audio Major,146


In [133]:
final_rating.drop_duplicates(['id','name'], keep='first',inplace=True)

In [134]:
final_rating.shape

(59850, 8)

In [135]:
final_rating.info()

<class 'pandas.core.frame.DataFrame'>
Index: 59850 entries, 0 to 487619
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   id          59850 non-null  int64 
 1   ISBN        59850 non-null  object
 2   rating      59850 non-null  int64 
 3   name        59850 non-null  object
 4   author      59850 non-null  object
 5   year        59850 non-null  object
 6   Publisher   59850 non-null  object
 7   num_rating  59850 non-null  int64 
dtypes: int64(3), object(5)
memory usage: 4.1+ MB


In [136]:
book_pivot=final_rating.pivot_table(columns='id',index='name',values='rating')

In [137]:
book_pivot.shape

(742, 888)

In [138]:
book_pivot.fillna(0,inplace=True)

In [139]:
book_pivot

id,254,2276,2766,2977,3363,3757,4017,4385,6242,6251,...,274004,274061,274301,274308,274808,275970,277427,277478,277639,278418
name,,,,,,,,,,,,,,,,,,,,,
1984,9.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1st to Die: A Novel,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2nd Chance,0.0,10.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4 Blondes,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
84 Charing Cross Road,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,10.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Year of Wonders,0.0,0.0,0.0,7.0,0.0,0.0,0.0,0.0,7.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
You Belong To Me,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Zen and the Art of Motorcycle Maintenance: An Inquiry into Values,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [34]:
from scipy.sparse import csr_matrix
book_sparse = csr_matrix(book_pivot)

In [35]:
type(book_sparse)

scipy.sparse._csr.csr_matrix

In [36]:
from sklearn.neighbors import NearestNeighbors
model = NearestNeighbors(algorithm = 'brute')

In [37]:
model.fit(book_sparse)

,n_neighbors,5
,radius,1.0
,algorithm,'brute'
,leaf_size,30
,metric,'minkowski'
,p,2
,metric_params,None
,n_jobs,None


In [38]:
distances, suggestions = model.kneighbors(book_pivot.iloc[237,:].values.reshape(1,-1),n_neighbors=6)

In [39]:
for i in suggestions: 
    print(book_pivot.index[i])

Index(['Harry Potter and the Chamber of Secrets (Book 2)',
       'Harry Potter and the Goblet of Fire (Book 4)',
       'Harry Potter and the Prisoner of Azkaban (Book 3)',
       'Harry Potter and the Sorcerer's Stone (Book 1)', 'Exclusive',
       'The Cradle Will Fall'],
      dtype='object', name='name')


In [40]:
distances[0][1].dtype

dtype('float64')

In [197]:
def recommonend_book(book_name):
    book_id = np.where(book_pivot.index==book_name)[0][0]
    distances, suggestions = model.kneighbors(book_pivot.iloc[book_id,:].values.reshape(1,-1),n_neighbors=6)
    for i in range(len(suggestions)):
        print(book_pivot.index[suggestions[i][1:]].tolist())

In [204]:
recommonend_book('Harry Potter and the Goblet of Fire (Book 4)')

['Harry Potter and the Prisoner of Azkaban (Book 3)', 'Harry Potter and the Order of the Phoenix (Book 5)', 'Harry Potter and the Chamber of Secrets (Book 2)', 'The Cradle Will Fall', 'Exclusive']


In [43]:
pkl.dump(model,open('model_book.pkl','wb'))

In [44]:
pkl.dump(recommonend_book,open('output_book.pkl','wb'))

In [45]:
book_pivot.to_csv('book_pivot.csv')

In [46]:
book.to_csv('final_books.csv')

In [47]:
book=book[['Book-Title','Image-URL-M']].rename(columns={'Book-Title':'name','Image-URL-M':'poster'})

In [154]:
book.drop_duplicates()

,name,poster
0,Classical Mythology,http://images.amazon.com/images/P/0195153448.0...
1,Clara Callan,http://images.amazon.com/images/P/0002005018.0...
2,Decision in Normandy,http://images.amazon.com/images/P/0060973129.0...
3,Flu: The Story of the Great Influenza Pandemic...,http://images.amazon.com/images/P/0374157065.0...
4,The Mummies of Urumchi,http://images.amazon.com/images/P/0393045218.0...
...,...,...
271355,There's a Bat in Bunk Five,http://images.amazon.com/images/P/0440400988.0...
271356,From One to One Hundred,http://images.amazon.com/images/P/0525447644.0...
271357,Lily Dale : The True Story of the Town that Ta...,http://images.amazon.com/images/P/006008667X.0...
271358,Republic (World's Classics),http://images.amazon.com/images/P/0192126040.0...


In [348]:
book[book['name']=="Republic (World's Classics)"]['poster'].values[0]

'http://images.amazon.com/images/P/0192126040.01.MZZZZZZZ.jpg'

In [349]:
book['poster'].isnull().sum()

np.int64(0)

In [351]:
book[book['name']== 'Jacob Have I Loved']['poster']

7702     http://images.amazon.com/images/P/0380564998.0...
8395     http://images.amazon.com/images/P/0590434985.0...
13005    http://images.amazon.com/images/P/0064403688.0...
51793    http://images.amazon.com/images/P/0690040784.0...
80739    http://images.amazon.com/images/P/0690040792.0...
92413    http://images.amazon.com/images/P/0064470598.0...
Name: poster, dtype: object

In [390]:
("http://images.amazon.com/images/P/0192126040.01.MZZZZZZZ.jpg").split('http')

['', '://images.amazon.com/images/P/0192126040.01.MZZZZZZZ.jpg']

In [383]:
list[0] = 'https'

In [389]:
"".join(list)

'https://images.amazon.com/images/P/0192126040.01.MZZZZZZZ.jpg'

In [393]:
def change(value):
    list = value.split('http')
    list[0]='https'
    return "".join(list)
    

In [395]:
book['poster']=book['poster'].apply(change)

In [49]:
book

,name,poster
0,Classical Mythology,http://images.amazon.com/images/P/0195153448.0...
1,Clara Callan,http://images.amazon.com/images/P/0002005018.0...
2,Decision in Normandy,http://images.amazon.com/images/P/0060973129.0...
3,Flu: The Story of the Great Influenza Pandemic...,http://images.amazon.com/images/P/0374157065.0...
4,The Mummies of Urumchi,http://images.amazon.com/images/P/0393045218.0...
...,...,...
271355,There's a Bat in Bunk Five,http://images.amazon.com/images/P/0440400988.0...
271356,From One to One Hundred,http://images.amazon.com/images/P/0525447644.0...
271357,Lily Dale : The True Story of the Town that Ta...,http://images.amazon.com/images/P/006008667X.0...
271358,Republic (World's Classics),http://images.amazon.com/images/P/0192126040.0...


In [184]:
book.drop_duplicates()

,name,poster
0,Classical Mythology,http://images.amazon.com/images/P/0195153448.0...
1,Clara Callan,http://images.amazon.com/images/P/0002005018.0...
2,Decision in Normandy,http://images.amazon.com/images/P/0060973129.0...
3,Flu: The Story of the Great Influenza Pandemic...,http://images.amazon.com/images/P/0374157065.0...
4,The Mummies of Urumchi,http://images.amazon.com/images/P/0393045218.0...
...,...,...
271355,There's a Bat in Bunk Five,http://images.amazon.com/images/P/0440400988.0...
271356,From One to One Hundred,http://images.amazon.com/images/P/0525447644.0...
271357,Lily Dale : The True Story of the Town that Ta...,http://images.amazon.com/images/P/006008667X.0...
271358,Republic (World's Classics),http://images.amazon.com/images/P/0192126040.0...


In [185]:
df =rating.merge(books.drop('year',axis=1),on='ISBN')

In [187]:
final_books=book.merge(df,on='name').drop_duplicates('name').drop(['id','ISBN','rating'],axis=1).sort_values('name')

In [189]:
final_books

,name,poster,author,Publisher
335993,A Light in the Storm: The Civil War Diary of ...,http://images.amazon.com/images/P/0590567330.0...,Karen Hesse,Hyperion Books for Children
871288,Always Have Popsicles,http://images.amazon.com/images/P/0964147726.0...,Rebecca Harvin,Rebecca L. Harvin
650336,Apple Magic (The Collector's series),http://images.amazon.com/images/P/0942320093.0...,Martina Boudreau,Amer Cooking Guild
262279,Beyond IBM: Leadership Marketing and Finance ...,http://images.amazon.com/images/P/0962295701.0...,Lou Mobley,"Teleonet, Incorporated"
802539,Clifford Visita El Hospital (Clifford El Gran...,http://images.amazon.com/images/P/0439188970.0...,Norman Bridwell,Scholastic
...,...,...,...,...
681232,Ã?Â?ber die Pflicht zum Ungehorsam gegen den S...,http://images.amazon.com/images/P/3257700512.0...,Henry David Thoreau,Diogenes
687039,Ã?Â?lpiraten.,http://images.amazon.com/images/P/3499232499.0...,Janwillem van de Wetering,Rowohlt Tb.
469798,Ã?Â?rger mit Produkt X. Roman.,http://images.amazon.com/images/P/325721538X.0...,Joan Aiken,Diogenes Verlag
568532,Ã?Â?stlich der Berge.,http://images.amazon.com/images/P/3442725739.0...,David Guterson,btb


In [190]:
final_books.to_csv('finalbooks.csv')

In [252]:
def recommend_book(book_name):
        book_id = np.where(book_pivot.index == book_name)[0][0]
        distances, suggestions = model.kneighbors(
            book_pivot.iloc[book_id, :].values.reshape(1, -1),
            n_neighbors=6
        )
        recommended_books = []
        for i in suggestions[0]:
            recommended_books.append(book_pivot.index[i])
        return recommended_books[1:]

book_name = 'Harry Potter and the Chamber of Secrets (Book 2)'
list = recommend_book(book_name)

poster_list=[]
name_list=[]
author_list=[]
publser_list=[]
for book in list:
        poster = final_books[final_books['name'] == book]['poster'].values[0]
        name = final_books[final_books['name'] == book]['name'].values[0]
        author = final_books[final_books['name'] == book]['author'].values[0]
        publisher = final_books[final_books['name'] == book]['Publisher'].values[0]
        poster_list.append(poster)
        name_list.append(name)
        author_list.append(author)
        publser_list.append(publisher)
print(name_list)

['Harry Potter and the Goblet of Fire (Book 4)', 'Harry Potter and the Prisoner of Azkaban (Book 3)', "Harry Potter and the Sorcerer's Stone (Book 1)", 'Exclusive', 'The Cradle Will Fall']


In [220]:
np.where(book_pivot.index =='Harry Potter and the Chamber of Secrets (Book 2)')

(array([237]),)

In [262]:
pd.read_csv('book_pivot.csv',index_col=0).index.tolist()

['1984',
 '1st to Die: A Novel',
 '2nd Chance',
 '4 Blondes',
 '84 Charing Cross Road',
 'A Bend in the Road',
 'A Case of Need',
 'A Child Called \\It\\": One Child\'s Courage to Survive"',
 'A Civil Action',
 'A Cry In The Night',
 'A Darkness More Than Night',
 'A Day Late and a Dollar Short',
 'A Fine Balance',
 'A Great Deliverance',
 'A Heartbreaking Work of Staggering Genius',
 'A Is for Alibi (Kinsey Millhone Mysteries (Paperback))',
 'A Lesson Before Dying (Vintage Contemporaries (Paperback))',
 'A Man Named Dave: A Story of Triumph and Forgiveness',
 'A Man in Full',
 'A Map of the World',
 'A Painted House',
 'A Patchwork Planet',
 'A Prayer for Owen Meany',
 'A Thin Dark Line (Mysteries &amp; Horror)',
 "A Thousand Acres (Ballantine Reader's Circle)",
 'A Time to Kill',
 "A Virtuous Woman (Oprah's Book Club (Paperback))",
 'A Walk to Remember',
 'A Widow for One Year',
 'A Wrinkle In Time',
 'A Wrinkle in Time',
 'A Year in Provence',
 "ANGELA'S ASHES",
 'Abduction',
 'Abou